# 03 — ACDC-Lite: Node-level Circuit Discovery

Load the latest `acdc_lite` run, render the ranked node table, plot
importance vs (layer, head) as a heatmap, and render the GraphViz dot
inline.  Notes on simplifications vs full edge-level ACDC.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path("__file__").parent.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

In [ ]:
from mech_interp.storage.sqlite_store import SQLiteResultStore

store = SQLiteResultStore(
    database_path=repo_root / "artifacts" / "results.db",
    artifact_dir=repo_root / "artifacts",
)

runs = store.list_runs(limit=100)
lite_runs = [r for r in runs if r.family == "acdc_lite"]

if not lite_runs:
    raise RuntimeError(
        "No acdc_lite runs found. "
        "Run: mech run --name acdc_lite"
    )

latest = lite_runs[0]
result = store.get_result(latest.id)
print(f"Run {latest.id} | {latest.spec_name} | status={latest.status}")
print("Metrics:", json.dumps(result.metrics if result else {}, indent=2))

## Ranked node table

Each row is a circuit node (attention head or MLP block).  `importance` is
the logit-diff drop caused by ablating that node — higher means the node is
more critical for the task.

In [ ]:
if result is None:
    raise RuntimeError(f"No result stored for run {latest.id}")

circuit_json_path = result.artifacts.get("circuit_json")
if circuit_json_path and Path(circuit_json_path).exists():
    circuit = json.loads(Path(circuit_json_path).read_text())
    nodes = circuit.get("nodes", [])
    df = pd.DataFrame(nodes)
    if "importance" in df.columns:
        df = df.sort_values("importance", ascending=False)
    print(df.to_string(index=False))
else:
    print("circuit_json artifact not present — showing metrics:")
    print(json.dumps(result.metrics, indent=2))
    nodes = []
    df = pd.DataFrame()

## Importance heatmap: layer × head

Rows = layers, columns = heads.  MLP nodes are shown in a separate column.
Brighter cells = more important nodes.  This view makes it easy to spot
which layers carry most of the circuit's causal weight.

In [ ]:
if df.empty or "layer" not in df.columns:
    print("Not enough structured node data to render heatmap.")
else:
    attn_df = df[df["component"] == "attn"].copy() if "component" in df.columns else df.copy()
    if not attn_df.empty and "head" in attn_df.columns and "importance" in attn_df.columns:
        n_layers = int(attn_df["layer"].max()) + 1
        n_heads = int(attn_df["head"].max()) + 1
        grid = np.zeros((n_layers, n_heads))
        for _, row in attn_df.iterrows():
            grid[int(row["layer"]), int(row["head"])] = float(row["importance"])

        fig, ax = plt.subplots(figsize=(max(6, n_heads), max(3, n_layers)))
        im = ax.imshow(grid, aspect="auto", cmap="viridis")
        ax.set_xlabel("Head")
        ax.set_ylabel("Layer")
        ax.set_title("ACDC-lite node importance (attention heads)")
        ax.set_xticks(range(n_heads))
        ax.set_yticks(range(n_layers))
        plt.colorbar(im, ax=ax, label="importance")
        plt.tight_layout()
        plt.show()
    else:
        print("Node data missing head/importance columns — skipping heatmap.")

## GraphViz circuit (inline)

The `.dot` file encodes the surviving circuit as a directed graph.
Green edges = surviving (importance ≥ τ), grey dashed = pruned.

In [ ]:
dot_path = result.artifacts.get("circuit_dot")
if dot_path and Path(dot_path).exists():
    dot_src = Path(dot_path).read_text()
    print(dot_src[:2000])
    try:
        import graphviz  # type: ignore[import]
        src = graphviz.Source(dot_src)
        src.render(filename="/tmp/acdc_lite_circuit", format="png", cleanup=True)
        from IPython.display import Image, display  # type: ignore[import]
        display(Image("/tmp/acdc_lite_circuit.png"))
    except ImportError:
        print("graphviz Python package not installed — printed dot source above.")
else:
    print("circuit_dot artifact not found.")

## ACDC-lite vs full edge-level ACDC

| Feature | acdc_lite | acdc_edge |
|---|---|---|
| Granularity | Node (whole head or MLP) | Directed edge (src → dst) |
| Passes per prompt | 1 per node | 1 + unique_srcs (optimised) |
| Approximation | Ablate node globally | Ablate src globally, weight by 1/layer_gap |
| Circuit output | Surviving nodes | Surviving edges + faithfulness |

Edge-level ACDC (`acdc_edge`) catches cases where a head is important only
for a specific downstream node — something node-level scoring misses.  Use
`acdc_lite` for fast iteration and `acdc_edge` for final circuit reports.